<a href="https://colab.research.google.com/github/EmmanuelMD3/Archivos-Java-/blob/master/PLN_b%C3%A1sico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Introducción práctica a PLN en Python**
# Autor: Jonathan Rojas Simón (jrojass@uaemex.mx)
Este notebook muestra un flujo completo de Procesamiento de Lenguaje Natural (PLN) en español, desde el preprocesamiento hasta la clasificación de sentimientos.

Los pasos (pipeline) que realiza este notebook son los siguientes:


1.   Tokenización y normalización básica
2.   Eliminación de stopwords
3.   Lematización con spaCy
4.   Representación vectorial (TF-IDF)
5.   Clasificación de sentimientos
6.   Clasificación interactiva del usuario




In [1]:
# Requisitos (instalar si no los tienes):
!pip install nltk scikit-learn spacy imbalanced-learn
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 94.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# **PASO 1: Tokenización y normalización básica**
En este paso es necesario separar los textos en elementos individuales (tokens) para después normalizarlos (pasar a minúsculas)

In [ ]:
# Se deben descargar todas las bibliotecas y recursos necesarios
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Es necesario probar con un texto de ejemplo
texto = "El procesamiento de lenguaje natural permite que las máquinas entiendan texto humano."
print("Texto original:", texto)
tokens = word_tokenize(texto.lower(), language='spanish')
print("Tokens:", tokens)

Texto original: El procesamiento de lenguaje natural permite que las máquinas entiendan texto humano.
Tokens: ['el', 'procesamiento', 'de', 'lenguaje', 'natural', 'permite', 'que', 'las', 'máquinas', 'entiendan', 'texto', 'humano', '.']


# **PASO 2: Eliminación de stopwords**
Una vez normalizado el texto, se deben filtrar las palabras o términos que no aportan algún significado (stopwords). Esto permite a los modelos mejorar su entendimiento.

In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords')
stopwords_es = set(stopwords.words('spanish'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
tokens_filtrados = [t for t in tokens if t.isalpha() and t not in stopwords_es]
print("Tokens filtrados (sin stopwords):", tokens_filtrados)

Tokens filtrados (sin stopwords): ['procesamiento', 'lenguaje', 'natural', 'permite', 'máquinas', 'entiendan', 'texto', 'humano']


# **PASO 3: Lematización con spaCy**
Ahora reducimos las palabras a su forma base (lemma), lo cual ayuda al modelo a generalizar mejor.

In [ ]:
import spacy
nlp = spacy.load("es_core_news_sm")
doc = nlp(" ".join(tokens_filtrados))
lemmas = [token.lemma_ for token in doc]
print("Lemmas:", lemmas)

Lemmas: ['procesamiento', 'lenguaje', 'natural', 'permitir', 'máquina', 'entender', 'texto', 'humano']


# **PASO 4: Representación vectorial (TF-IDF)**
Se aplica el pesado TF-IDF a un conjunto de textos más amplio para capturar relaciones de importancia entre oraciones.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
corpus = [
"El perro juega en el parque",
"La niña juega con su perro",
"El perro ladra",
"Los gatos duermen todo el día",
"El perro corre tras la pelota"
]

In [ ]:
# Preprocesamos el corpus aplicando tokenización, filtrado y lematización
corpus_limpio = []
for doc_text in corpus:
  doc = nlp(doc_text.lower())
  tokens = [t.lemma_ for t in doc if t.is_alpha and t.text not in stopwords_es]
  corpus_limpio.append(" ".join(tokens))

In [ ]:
corpus_limpio

['perro jugar parque',
 'niña jugar perro',
 'perro ladra',
 'gato duermen día',
 'perro correr tras pelota']

In [ ]:
vec_demo = TfidfVectorizer()
X_demo = vec_demo.fit_transform(corpus_limpio)
print("TF-IDF vocabulario:", vec_demo.vocabulary_)
print("Matriz TF-IDF:\n", X_demo.toarray())

TF-IDF vocabulario: {'perro': 9, 'jugar': 4, 'parque': 7, 'niña': 6, 'ladra': 5, 'gato': 3, 'duermen': 1, 'día': 2, 'correr': 0, 'tras': 10, 'pelota': 8}
Matriz TF-IDF:
 [[0.         0.         0.         0.         0.57506256 0.
  0.         0.71277522 0.         0.40156512 0.        ]
 [0.         0.         0.         0.         0.57506256 0.
  0.71277522 0.         0.         0.40156512 0.        ]
 [0.         0.         0.         0.         0.         0.87124678
  0.         0.         0.         0.49084524 0.        ]
 [0.         0.57735027 0.57735027 0.57735027 0.         0.
  0.         0.         0.         0.         0.        ]
 [0.54903633 0.         0.         0.         0.         0.
  0.         0.         0.54903633 0.30931749 0.54903633]]


# **PASO 5: Clasificación de sentimientos**
Entrenamos un modelo de clasificación (Regresión logística) con un corpus ampliado y preprocesado.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
import numpy as np

In [ ]:
textos = [
# Positivos
"Me encanta este producto, es buenísimo!",
"Excelente calidad, lo recomiendo totalmente",
"Muy satisfecho con mi compra",
"Estoy muy feliz con el resultado",
"Increíble atención y producto de gran calidad",
"Maravilloso servicio, volveré a comprar",
"El artículo funciona perfectamente",
"Me alegra haberlo comprado, fue una buena decisión",
"Todo salió excelente, lo amo!",
"Muy buena experiencia y atención",
# Negativos
"No me gustó, es terrible y muy malo",
"Horrible experiencia, no volvería a comprar",
"El peor servicio que he tenido",
"Este artículo es pésimo y defectuoso",
"No funcionó, estoy decepcionado",
"La calidad es muy mala, me arrepiento",
"Atención deficiente y producto roto",
"Fue una pérdida de dinero",
"Mala experiencia, no lo recomiendo",
"Demasiado caro para lo que ofrece"
]


etiquetas = [1]*10 + [0]*10 # Balance 50/50

In [ ]:
# Preprocesamiento
def preprocesar_texto(texto):
  doc = nlp(texto.lower())
  tokens = [tok.lemma_ for tok in doc if tok.is_alpha and tok.text not in stopwords_es]
  return " ".join(tokens)

In [ ]:
# Preprocesamos los textos
textos_limpios = [preprocesar_texto(t) for t in textos]

In [ ]:
textos_limpios

['encantar producto buenísimo',
 'excelente calidad recomeir totalmente',
 'satisfecho compra',
 'feliz resultado',
 'increíble atención producto gran calidad',
 'maravilloso servicio volver comprar',
 'artículo funcionar perfectamente',
 'alegrar haber él comprar buen decisión',
 'salir excelente amar',
 'buen experiencia atención',
 'gustar terrible malo',
 'horrible experiencia volver comprar',
 'peor servicio',
 'artículo pésimo defectuoso',
 'funcionar decepcionado',
 'calidad malo arrepientir',
 'atención deficiente producto roto',
 'pérdida dinero',
 'malo experiencia recomeir',
 'demasiado caro ofrecer']

In [ ]:
# Vectorización
vec = TfidfVectorizer(ngram_range=(1,2), min_df=1)
X = vec.fit_transform(textos_limpios)

In [ ]:
# División de datos
X_train, X_test, y_train, y_test = train_test_split(X, etiquetas, test_size=0.3, random_state=42)

# Entrenamiento del modelo
clf = LogisticRegression(max_iter=300)
clf.fit(X_train, y_train)

LogisticRegression(max_iter=300)

In [ ]:
# Evaluación
preds = clf.predict(X_test)
print("\nExactitud del modelo balanceado:", accuracy_score(y_test, preds))
print("Reporte de clasificación:\n", classification_report(y_test, preds))
print("Matriz de confusión:\n", confusion_matrix(y_test, preds))


Exactitud del modelo balanceado: 0.3333333333333333
Reporte de clasificación:
               precision    recall  f1-score   support

           0       0.33      1.00      0.50         2
           1       0.00      0.00      0.00         4

    accuracy                           0.33         6
   macro avg       0.17      0.50      0.25         6
weighted avg       0.11      0.33      0.17         6

Matriz de confusión:
 [[2 0]
 [4 0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# **PASO 6: Clasificación interactiva del usuario**
Integramos todos los pasos: tokenización, limpieza, lematización y clasificación final.

In [ ]:
print("\nPrueba la clasificación de tus propios textos:")
while True:
  entrada = input("Escribe una reseña o comentario (o 'salir' para terminar): ")
  if entrada.lower() == 'salir':
    print("Fin de la sesión interactiva.")
    break

  # Preprocesamiento del texto introducido
  doc = nlp(entrada.lower())
  tokens = [tok.lemma_ for tok in doc if tok.is_alpha and tok.text not in stopwords_es]
  texto_procesado = " ".join(tokens)

  # Vectorización y predicción
  X_nuevo = vec.transform([texto_procesado])
  pred = clf.predict(X_nuevo)[0]
  prob = clf.predict_proba(X_nuevo)[0][pred]
  sentimiento = 'Positivo' if pred == 1 else 'Negativo'
  print(f"Sentimiento estimado: {sentimiento} (confianza: {prob:.2f})\n")


Prueba la clasificación de tus propios textos:
Sentimiento estimado: Negativo (confianza: 0.58)

Sentimiento estimado: Positivo (confianza: 0.54)

Sentimiento estimado: Negativo (confianza: 0.58)

Sentimiento estimado: Negativo (confianza: 0.58)

Sentimiento estimado: Negativo (confianza: 0.58)

Sentimiento estimado: Negativo (confianza: 0.57)

Sentimiento estimado: Positivo (confianza: 0.54)

Sentimiento estimado: Negativo (confianza: 0.57)

Sentimiento estimado: Negativo (confianza: 0.57)

Sentimiento estimado: Negativo (confianza: 0.57)

Sentimiento estimado: Negativo (confianza: 0.57)

Sentimiento estimado: Negativo (confianza: 0.50)

